In [5]:
import pandas as pd
import numpy as np
import pandas_datareader as pdr
from datetime import datetime
import os


In [6]:
print("\n1. Loading existing data...")
df = pd.read_csv(r"C:\\Users\\user\\OneDrive\\Desktop\\financial_regime_clasification\\data\\processed\\xau_with_vix_dxy.csv", parse_dates=['date'])
df.set_index('date', inplace=True)


1. Loading existing data...


In [7]:
print(f"   Original data: {len(df)} rows")
print(f"   Date range: {df.index.min()} to {df.index.max()}")

# 2. Get date range
start_date = df.index.min()
end_date = df.index.max()

   Original data: 2942 rows
   Date range: 2025-04-22 03:00:00 to 2025-10-10 15:00:00


In [ ]:
print(f"\n2. Downloading macro data from {start_date.date()} to {end_date.date()}...")

# 3. Download macro features
macro_data = pd.DataFrame(index=df.index)

try:
    # TNX: 10-Year Treasury Yield (inverse correlation with gold)
    print("   Downloading TNX (10-Year Treasury Yield)...")
    tnx = pdr.DataReader('DGS10', 'fred', start_date, end_date)
    macro_data['tnx'] = tnx['DGS10'] / 100  # Convert to decimal (e.g., 4.2% -> 0.042)
    print(f" Got {len(tnx)} rows")
    
    # TIPS: 10-Year Real Interest Rate (gold loves negative real rates)
    print("   Downloading TIPS Real Rate (DFII10)...")
    tips = pdr.DataReader('DFII10', 'fred', start_date, end_date)
    macro_data['real_rate'] = tips['DFII10'] / 100  # Convert to decimal
    print(f" Got {len(tips)} rows")
    
    # Breakeven Inflation (TNX - TIPS = market inflation expectation)
    print("   Calculating breakeven inflation...")
    macro_data['breakeven_inflation'] = macro_data['tnx'] - macro_data['real_rate']
    print(" Calculated")
    
except Exception as e:
    print(f" Download error: {e}")
    print("   Trying alternative data sources...")
    
    # Alternative: Use different FRED series if main ones fail
    try:
        tnx = pdr.DataReader('DGS10', 'fred', start_date, end_date)
        macro_data['tnx'] = tnx['DGS10'] / 100
    except:
        print(" TNX unavailable, using forward fill from existing")
        macro_data['tnx'] = np.nan
    
    try:
        # Alternative TIPS series
        tips = pdr.DataReader('FII10', 'fred', start_date, end_date)
        macro_data['real_rate'] = tips['FII10'] / 100
    except:
        print(" TIPS unavailable")
        macro_data['real_rate'] = np.nan

  


2. Downloading macro data from 2025-04-22 to 2025-10-10...
      ✓ Got 123 rows
      ✓ Got 123 rows
   Calculating breakeven inflation...
      ✓ Calculated


In [ ]:
# 4. Add derived features
print("\n3. Creating derived features...")

# Gold vs Yield inverse relationship (strong signal)
macro_data['gold_yield_spread'] = macro_data['tnx'] - macro_data['real_rate']

# Rate of change (momentum signals)
macro_data['tnx_change_5d'] = macro_data['tnx'].diff(5)
macro_data['real_rate_change_5d'] = macro_data['real_rate'].diff(5)

# Negative real rate indicator (binary: 1 if real rate < 0)
macro_data['negative_real_rate'] = (macro_data['real_rate'] < 0).astype(int)

print("Created derived features")



3. Creating derived features...
   ✓ Created derived features


In [10]:
# 5. Merge with your data
print("\n4. Merging with XAUUSD data...")
combined_df = df.copy()
macro_features = ['tnx', 'real_rate', 'breakeven_inflation', 
                  'gold_yield_spread', 'tnx_change_5d', 
                  'real_rate_change_5d', 'negative_real_rate']

for feature in macro_features:
    if feature in macro_data.columns:
        combined_df[feature] = macro_data[feature]
    else:
        combined_df[feature] = np.nan



4. Merging with XAUUSD data...


In [ ]:
# 6. Handle missing values (weekends/holidays)
print("\n5. Handling missing values...")
before_missing = combined_df[macro_features].isnull().sum().sum()
combined_df[macro_features] = combined_df[macro_features].fillna(method='ffill')
combined_df[macro_features] = combined_df[macro_features].fillna(method='bfill')
combined_df[macro_features] = combined_df[macro_features].fillna(0)
after_missing = combined_df[macro_features].isnull().sum().sum()

print(f" Missing values filled: {before_missing} → {after_missing}")



5. Handling missing values...
   Missing values filled: 17184 → 0


C:\Users\user\AppData\Local\Temp\ipykernel_28520\3232135672.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_df[macro_features] = combined_df[macro_features].fillna(method='ffill')
C:\Users\user\AppData\Local\Temp\ipykernel_28520\3232135672.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_df[macro_features] = combined_df[macro_features].fillna(method='bfill')


In [15]:
# Add these instead of 5-day changes:
combined_df['tnx_change_24h'] = combined_df['tnx'].diff(24)  # 1 day change
combined_df['real_rate_change_24h'] = combined_df['real_rate'].diff(24)
combined_df['tnx_change_168h'] = combined_df['tnx'].diff(168)  # 1 week change

# Fill NaN with 0
combined_df[['tnx_change_24h', 'real_rate_change_24h', 'tnx_change_168h']] = \
    combined_df[['tnx_change_24h', 'real_rate_change_24h', 'tnx_change_168h']].fillna(0)

In [ ]:
# 7. Save to new file
print("\n6. Saving enriched data...")
output_path = r"C:\\Users\\user\\OneDrive\\Desktop\\financial_regime_clasification\\data\\processed\\xau_with_all_macro.csv"
combined_df.to_csv(output_path)
print(f" Saved to: {output_path}")



6. Saving enriched data...
   ✓ Saved to: C:\\Users\\user\\OneDrive\\Desktop\\financial_regime_clasification\\data\\processed\\xau_with_all_macro.csv


In [ ]:
# 8. Summary statistics
print("\n" + "="*80)
print("FEATURE SUMMARY")
print("="*80)

print(f"\nNew macro features added:")
for feature in macro_features:
    if feature in combined_df.columns:
        non_null = combined_df[feature].notnull().sum()
        print(f"  • {feature}: {non_null}/{len(combined_df)} rows (mean={combined_df[feature].mean():.4f})")

print(f"\nFinal dataset shape: {combined_df.shape}")
print(f"Columns: {combined_df.columns.tolist()}")

print("\n" + "="*80)
print(" MACRO FEATURES ADDED SUCCESSFULLY!")
print("="*80)



FEATURE SUMMARY

New macro features added:
  • tnx: 2942/2942 rows (mean=0.0430)
  • real_rate: 2942/2942 rows (mean=0.0196)
  • breakeven_inflation: 2942/2942 rows (mean=0.0234)
  • gold_yield_spread: 2942/2942 rows (mean=0.0234)
  • tnx_change_5d: 2942/2942 rows (mean=0.0000)
  • real_rate_change_5d: 2942/2942 rows (mean=0.0000)
  • negative_real_rate: 2942/2942 rows (mean=0.0000)

Final dataset shape: (2942, 28)
Columns: ['Unnamed: 0', 'open', 'high', 'low', 'close', 'volume', 'past_ret_1h', 'past_ret_2h', 'past_ret_4h', 'past_ret_8h', 'past_ret_16h', 'forward regime', 'probability', '0.6 percent prediction', '1 percent prediction', '1.5 percent prediction', 'vix', 'dxy', 'tnx', 'real_rate', 'breakeven_inflation', 'gold_yield_spread', 'tnx_change_5d', 'real_rate_change_5d', 'negative_real_rate', 'tnx_change_24h', 'real_rate_change_24h', 'tnx_change_168h']

✓ MACRO FEATURES ADDED SUCCESSFULLY!


In [ ]:
# 9. Quick correlation check with gold
print("\n7. Correlation with Gold Close Price:")
correlations = combined_df[macro_features + ['close']].corr()['close'].sort_values(ascending=False)
print(correlations.to_string())

# Verify key relationships (should be negative for yields)
tnx_corr = combined_df['close'].corr(combined_df['tnx'])
print(f"\n✓ Gold vs TNX correlation: {tnx_corr:.3f} (expected negative)")
if tnx_corr < 0:
    print(" Correct inverse relationship!")
else:
    print(" Unexpected positive correlation - check data")


7. Correlation with Gold Close Price:
close                  1.000000
breakeven_inflation    0.173817
gold_yield_spread      0.173817
real_rate             -0.720177
tnx                   -0.724865
tnx_change_5d               NaN
real_rate_change_5d         NaN
negative_real_rate          NaN

✓ Gold vs TNX correlation: -0.725 (expected negative)
  ✅ Correct inverse relationship!
